# Validate the Type A NPZ files and dataset split

This notebook:
- Finds the processed Type A SFINCS NPZ files
- Inspects one example NPZ file
- Validates every NPZ file's arrays, metadata, and multiscale grid dimensions
- Loads and verifies the five-way dataset_split_type_A.json already built
  by the SFINCS-to-npz notebook (does NOT regenerate the split here)

### Imports

In [1]:
from pathlib import Path  # Handle project and dataset paths.
import json  # Read NPZ metadata and save the dataset split.

import numpy as np  # Read and validate the NPZ arrays.

### Paths

In [2]:
PROJECT_ROOT = Path(r"C:\Users\Prof. KN Jha\Desktop\SFINCS_mSWE-GNN_v3")

NPZ_FOLDER = PROJECT_ROOT / "outputs" / "batch_type_A_training_npz"
DATASET_SPLIT_PATH = NPZ_FOLDER / "dataset_split_type_A.json"  # Same location the SFINCS-to-npz notebook saved to.

print("NPZ folder:", NPZ_FOLDER)
print("NPZ folder exists:", NPZ_FOLDER.exists())
print("Dataset split file:", DATASET_SPLIT_PATH)
print("Dataset split file exists:", DATASET_SPLIT_PATH.exists())

NPZ folder: C:\Users\Prof. KN Jha\Desktop\SFINCS_mSWE-GNN_v3\outputs\batch_type_A_training_npz
NPZ folder exists: True
Dataset split file: C:\Users\Prof. KN Jha\Desktop\SFINCS_mSWE-GNN_v3\outputs\batch_type_A_training_npz\dataset_split_type_A.json
Dataset split file exists: True


## Find the Type A NPZ files

In [4]:
npz_files = sorted(NPZ_FOLDER.glob("*.npz"))
print("Number of NPZ files:", len(npz_files))
print()

for npz_path in npz_files:
    print(npz_path.name)

Number of NPZ files: 276

DEM_10002_L3_RP002_temporal_uniform_type_A.npz
DEM_10002_L3_RP005_temporal_uniform_type_A.npz
DEM_10002_L3_RP010_temporal_uniform_type_A.npz
DEM_10002_L3_RP020_temporal_uniform_type_A.npz
DEM_10002_L3_RP050_temporal_uniform_type_A.npz
DEM_10002_L3_RP100_temporal_uniform_type_A.npz
DEM_10002_sloped_5m_80deg_L3_RP002_temporal_uniform_type_A.npz
DEM_10002_sloped_5m_80deg_L3_RP005_temporal_uniform_type_A.npz
DEM_10002_sloped_5m_80deg_L3_RP010_temporal_uniform_type_A.npz
DEM_10002_sloped_5m_80deg_L3_RP020_temporal_uniform_type_A.npz
DEM_10002_sloped_5m_80deg_L3_RP050_temporal_uniform_type_A.npz
DEM_10002_sloped_5m_80deg_L3_RP100_temporal_uniform_type_A.npz
DEM_10012_L3_RP002_temporal_uniform_type_A.npz
DEM_10012_L3_RP005_temporal_uniform_type_A.npz
DEM_10012_L3_RP010_temporal_uniform_type_A.npz
DEM_10012_L3_RP020_temporal_uniform_type_A.npz
DEM_10012_L3_RP050_temporal_uniform_type_A.npz
DEM_10012_L3_RP100_temporal_uniform_type_A.npz
DEM_10014_L3_RP002_temporal_unif

## Validate every NPZ file

In [5]:
required_arrays = {"dem", "water_depth", "model_time_seconds", "rainfall_time_seconds", "rain_mmhr", "metadata"}
required_metadata = {"case_name", "dem_name", "rainfall_name", "rainfall_type", "dx", "dy", "runtime_seconds", "domain_role", "topography", "split"}

for npz_path in npz_files:
    with np.load(npz_path, allow_pickle=False) as npz:
        missing_arrays = required_arrays - set(npz.files)
        if missing_arrays:
            raise ValueError(f"{npz_path.name} is missing: {sorted(missing_arrays)}")

        dem = npz["dem"]
        water_depth = npz["water_depth"]
        model_time = npz["model_time_seconds"]
        rainfall_time = npz["rainfall_time_seconds"]
        rainfall = npz["rain_mmhr"]
        metadata = json.loads(npz["metadata"].item())

        missing_metadata = required_metadata - set(metadata)
        if missing_metadata:
            raise ValueError(f"{npz_path.name} is missing metadata: {sorted(missing_metadata)}")

        if dem.ndim != 2:
            raise ValueError(f"{npz_path.name} has an invalid DEM shape: {dem.shape}")
        if dem.shape[0] % 4 != 0 or dem.shape[1] % 4 != 0:
            raise ValueError(f"{npz_path.name} cannot be pooled twice: DEM shape = {dem.shape}")
        if water_depth.ndim != 3:
            raise ValueError(f"{npz_path.name} has an invalid water-depth shape: {water_depth.shape}")
        if water_depth.shape[1:] != dem.shape:
            raise ValueError(f"{npz_path.name} has inconsistent spatial dimensions")
        if water_depth.shape[0] != len(model_time):
            raise ValueError(f"{npz_path.name} has inconsistent model times")
        if len(rainfall) != len(rainfall_time):
            raise ValueError(f"{npz_path.name} has inconsistent rainfall times")

print("All NPZ files are valid:", len(npz_files))

All NPZ files are valid: 276


## Verify the existing dataset split 

The SFINCS-to-npz notebook already built and saved the five-way split.
This just cross-checks it against the files actually on disk, so any
NPZ that got added, removed, or renamed since the split was saved would
be caught here rather than silently causing a mismatch later.

In [8]:
with DATASET_SPLIT_PATH.open("r", encoding="utf-8") as file:
    dataset_split = json.load(file)

expected_splits = {"train", "validation", "test_unseen_rainfall", "test_unseen_domain", "test_unseen_domain_and_rainfall"}
actual_splits = set(dataset_split.keys())

if actual_splits != expected_splits:
    raise ValueError(f"Split keys mismatch. Expected {expected_splits}, got {actual_splits}")

all_split_files = [path for paths in dataset_split.values() for path in paths]

if len(all_split_files) != len(npz_files):
    raise ValueError(f"Split covers {len(all_split_files)} files but {len(npz_files)} npz files exist on disk.")

if len(all_split_files) != len(set(all_split_files)):
    raise ValueError("One or more npz files appear in multiple splits.")

for split_name, paths in dataset_split.items():
    print(f"{split_name}: {len(paths)} files")

print("Dataset split verified against files on disk.")

train: 120 files
validation: 30 files
test_unseen_rainfall: 30 files
test_unseen_domain: 80 files
test_unseen_domain_and_rainfall: 16 files
Dataset split verified against files on disk.


### Converting one npz into PyTorch Graph

In [9]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))

from scripts.sfincs_dataset_conversion.pytorch_geometric import convert_one_npz_to_graph

example_path = PROJECT_ROOT / dataset_split["train"][0]
example_graph = convert_one_npz_to_graph(example_path)

print("case_name:", example_graph.case_name)
print("fine nodes:", example_graph.num_fine_nodes)
print("medium nodes:", example_graph.num_medium_nodes)
print("coarse nodes:", example_graph.num_coarse_nodes)
print("water_depth shape:", example_graph.water_depth.shape)
print("rainfall shape:", example_graph.rainfall.shape)

c:\Users\Prof. KN Jha\miniconda3\envs\hydromt-sfincs\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


case_name: DEM_41_L3_RP002_temporal_uniform_type_A
fine nodes: 4096
medium nodes: 1024
coarse nodes: 256
water_depth shape: torch.Size([4096, 91])
rainfall shape: torch.Size([4096, 91])


### Converting all NPZ files to graphs

In [10]:
import torch
from tqdm import tqdm

GRAPHS_FOLDER = PROJECT_ROOT / "outputs" / "batch_type_A_complete_graphs"
GRAPHS_FOLDER.mkdir(parents=True, exist_ok=True)

def load_split_graphs(split_name, relative_paths):
    """Convert every npz file in one split into a complete graph."""

    graphs = []
    for relative_path in tqdm(relative_paths, desc=split_name):
        npz_path = PROJECT_ROOT / relative_path
        graphs.append(convert_one_npz_to_graph(npz_path))
    return graphs

complete_graphs = {
    split_name: load_split_graphs(split_name, relative_paths)
    for split_name, relative_paths in dataset_split.items()
}

for split_name, graphs in complete_graphs.items():
    print(f"{split_name}: {len(graphs)} graphs built")

test_unseen_domain_and_rainfall: 100%|██████████| 16/16 [00:00<00:00, 18.01it/s]

train: 120 graphs built
validation: 30 graphs built
test_unseen_rainfall: 30 graphs built
test_unseen_domain: 80 graphs built
test_unseen_domain_and_rainfall: 16 graphs built


## Save the complete graphs

In [11]:
for split_name, graphs in complete_graphs.items():
    save_path = GRAPHS_FOLDER / f"{split_name}_graphs.pt"
    torch.save(graphs, save_path)
    print(f"Saved {split_name}: {len(graphs)} graphs -> {save_path}")

Saved train: 120 graphs -> C:\Users\Prof. KN Jha\Desktop\SFINCS_mSWE-GNN_v3\outputs\batch_type_A_complete_graphs\train_graphs.pt
Saved validation: 30 graphs -> C:\Users\Prof. KN Jha\Desktop\SFINCS_mSWE-GNN_v3\outputs\batch_type_A_complete_graphs\validation_graphs.pt
Saved test_unseen_rainfall: 30 graphs -> C:\Users\Prof. KN Jha\Desktop\SFINCS_mSWE-GNN_v3\outputs\batch_type_A_complete_graphs\test_unseen_rainfall_graphs.pt
Saved test_unseen_domain: 80 graphs -> C:\Users\Prof. KN Jha\Desktop\SFINCS_mSWE-GNN_v3\outputs\batch_type_A_complete_graphs\test_unseen_domain_graphs.pt
Saved test_unseen_domain_and_rainfall: 16 graphs -> C:\Users\Prof. KN Jha\Desktop\SFINCS_mSWE-GNN_v3\outputs\batch_type_A_complete_graphs\test_unseen_domain_and_rainfall_graphs.pt


### Reload and verify

In [14]:
reloaded_train_graphs = torch.load(GRAPHS_FOLDER / "train_graphs.pt", weights_only=False)

print("Reloaded train graphs:", len(reloaded_train_graphs))
print("First graph case_name:", reloaded_train_graphs[100].case_name)
print("First graph fine nodes:", reloaded_train_graphs[100].num_fine_nodes)
print("First graph medium nodes:", reloaded_train_graphs[100].num_medium_nodes)
print("First graph coarse nodes:", reloaded_train_graphs[100].num_coarse_nodes)
print("First graph water_depth shape:", reloaded_train_graphs[100].water_depth.shape)
print("First graph rainfall shape:", reloaded_train_graphs[100].rainfall.shape)

Reloaded train graphs: 120
First graph case_name: DEM_38_sloped_10m_243deg_L3_RP002_temporal_uniform_type_A
First graph fine nodes: 4096
First graph medium nodes: 1024
First graph coarse nodes: 256
First graph water_depth shape: torch.Size([4096, 91])
First graph rainfall shape: torch.Size([4096, 91])
